# Fine-tune MineGPT

Trains MineGPT's planner on the episodes your agent logged, with QLoRA on a free Colab GPU.

1. On your PC, in `brain/`: `python -m minegpt_brain.train export` makes `minegpt-data/minegpt-data.zip`.
2. Here: **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
3. At the end you get a `.gguf` file to run on your PC with llama.cpp (see `docs/training.md`).

A run on a T4 takes roughly 10 to 30 minutes per 100 episodes, plus about 10 minutes to build the GGUF file.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## Upload the dataset

Pick `minegpt-data.zip` when asked. (Or put it in Google Drive and set `FROM_DRIVE = True`.)

In [ ]:
FROM_DRIVE = False
DRIVE_ZIP = "/content/drive/MyDrive/minegpt/minegpt-data.zip"

import os, zipfile
if FROM_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    path = DRIVE_ZIP
else:
    from google.colab import files
    path = next(iter(files.upload()))
zipfile.ZipFile(path).extractall("/content/minegpt-data")
os.chdir("/content/minegpt-data")
print(open("info.json").read())

## Train

The defaults suit a T4: Qwen3 4B Instruct, 4-bit, LoRA rank 16, 2 epochs, samples up to 16k tokens.
With few episodes (under 100) try `EPOCHS = 3`. For the 8B model use `MODEL = "unsloth/Qwen3-8B"` and
`MAX_SEQ = 12288`; it is slower here and on your laptop.

In [ ]:
MODEL = "unsloth/Qwen3-4B-Instruct-2507"
EPOCHS = 2
MAX_SEQ = 16384
GGUF = "q4_k_m"   # "none" to skip, "q8_0" for a bigger, slightly better file

!python finetune.py --model {MODEL} --epochs {EPOCHS} --max-seq {MAX_SEQ} --gguf {GGUF} --out /content/minegpt

## Download

The GGUF file is a few GB, so copying it to Google Drive is often faster than a browser download.
The LoRA adapter (`lora.zip`, small) is worth keeping too, for training further later.

In [ ]:
import glob, shutil
ggufs = sorted(glob.glob("/content/**/*.gguf", recursive=True))
print(ggufs)
shutil.make_archive("/content/lora", "zip", "/content/minegpt/lora")

TO_DRIVE = True
if TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/minegpt", exist_ok=True)
    for f in ggufs + ["/content/lora.zip"]:
        shutil.copy(f, "/content/drive/MyDrive/minegpt/")
    print("Copied to MyDrive/minegpt")
else:
    from google.colab import files
    for f in ggufs + ["/content/lora.zip"]:
        files.download(f)